# 04 CV Quality Analysis

This notebook analyzes the quality of a CV independently from any specific job posting.

Main goals:
- load extracted CV text from the previous PDF extraction step
- use an LLM to analyze CV quality
- evaluate clarity, structure, completeness and IT relevance
- generate CV improvement recommendations
- calculate a transparent CV Quality Score
- save the result as JSON and Markdown report

This step does not compare the CV with a job posting.

## 1. Imports and Settings

In [1]:
import os
import json
import pandas as pd

from pathlib import Path
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import List

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

In [2]:
load_dotenv(dotenv_path=Path("../.env"))

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [3]:
PROCESSED_TEST_CV_DIR = Path("../data/processed/test_cv")
OUTPUT_CV_QUALITY_DIR = Path("../outputs/cv_quality")

cv_text_path = PROCESSED_TEST_CV_DIR / "test_cv_extracted_text.txt"

json_output_path = OUTPUT_CV_QUALITY_DIR / "cv_quality_analysis.json"
markdown_output_path = OUTPUT_CV_QUALITY_DIR / "cv_quality_report.md"

cv_text_path

WindowsPath('../data/processed/test_cv/test_cv_extracted_text.txt')

## 2. Load Extracted CV Text

In [4]:
with open(cv_text_path, "r", encoding="utf-8") as file:
    cv_text = file.read()

## 3. Basic Text Check

In [5]:
cv_text_stats = {
    "character_count": len(cv_text),
    "word_count": len(cv_text.split()),
    "line_count": len(cv_text.splitlines())
}


In [6]:
cv_text_stats

{'character_count': 2399, 'word_count': 301, 'line_count': 45}

## 4. Define Structured Output Schema

In [7]:
class CVQualityScores(BaseModel):

    structure_and_readability_score: int = Field(
        description="Score from 0 to 100 for CV structure, readability and section organization."
    )

    completeness_score: int = Field(
        description="Score from 0 to 100 for completeness of basic CV information."
    )

    technical_skills_clarity_score: int = Field(
        description="Score from 0 to 100 for clarity and organization of technical skills."
    )

    experience_description_score: int = Field(
        description="Score from 0 to 100 for clarity and detail of work experience descriptions."
    )

    projects_description_score: int = Field(
        description="Score from 0 to 100 for clarity and detail of project descriptions."
    )

    measurable_results_score: int = Field(
        description="Score from 0 to 100 for presence of measurable achievements and concrete results."
    )

    it_relevance_score: int = Field(
        description="Score from 0 to 100 for relevance of the CV to IT roles."
    )


class CVQualityAnalysis(BaseModel):

    overall_summary: str = Field(
        description="Short summary of the overall CV quality."
    )

    scores: CVQualityScores = Field(
        description="Individual CV quality scores."
    )

    strengths: List[str] = Field(
        description="Strong aspects of the CV that are clearly visible from the provided text."
    )

    weaknesses: List[str] = Field(
        description="Weak aspects of the CV based only on the provided text."
    )

    missing_or_unclear_sections: List[str] = Field(
        description="Sections or information that are missing, unclear or not detailed enough."
    )

    cv_improvement_recommendations: List[str] = Field(
        description="Practical and honest recommendations for improving CV clarity, structure and completeness."
    )

    evidence_notes: List[str] = Field(
        description="Short notes explaining which parts of the CV text support the analysis."
    )

## 5. Initialize OpenAI Model

In [8]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)


In [9]:
structured_llm = llm.with_structured_output(CVQualityAnalysis)

## 6. Create CV Quality Analysis Prompt

In [10]:
cv_quality_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are an AI assistant specialized in CV quality analysis for IT job candidates.

Your task is to analyze the quality of the provided CV independently from any specific job posting.

Important rules:
- Analyze only the information present in the CV text.
- Do not invent skills, experience, education, certifications, projects or achievements.
- Do not assume that the candidate has a skill if it is not visible in the CV.
- If something is unclear or only partially presented, explicitly say that it is unclear or partially presented.
- Focus on clarity, structure, completeness, technical skills presentation, experience descriptions, project descriptions and IT relevance.
- Recommendations should improve the CV honestly, without suggesting false information.
- The analysis should be useful for an IT candidate.

Return the analysis using the required structured schema.
"""
        ),
        (
            "human",
            """
Analyze the following CV text:

{cv_text}
"""
        )
    ]
)

## 7. Run CV Quality Analysis

In [11]:
cv_quality_chain = cv_quality_prompt | structured_llm

cv_quality_result = cv_quality_chain.invoke(
    {
        "cv_text": cv_text
    }
)

In [12]:
cv_quality_result

CVQualityAnalysis(overall_summary='The CV presents a strong candidate profile with a solid educational background and relevant project experience. However, it could benefit from improved structure and clarity in certain sections, particularly in the skills presentation and project descriptions.', scores=CVQualityScores(structure_and_readability_score=75, completeness_score=85, technical_skills_clarity_score=70, experience_description_score=80, projects_description_score=75, measurable_results_score=60, it_relevance_score=90), strengths=['Strong academic performance with a high GPA (9.2/10.0)', 'Relevant coursework that aligns well with software engineering roles', 'Diverse project experience showcasing practical application of skills', 'Clear communication of technical skills and areas of specialization'], weaknesses=['Skills section lacks organization and clarity, making it hard to quickly assess proficiency', 'Project descriptions could include measurable outcomes or specific technol

## 8. Convert Structured Result to Dictionary

In [13]:
if hasattr(cv_quality_result, "model_dump"):
    cv_quality_dict = cv_quality_result.model_dump()
else:
    cv_quality_dict = cv_quality_result.dict()


In [14]:
cv_quality_dict

{'overall_summary': 'The CV presents a strong candidate profile with a solid educational background and relevant project experience. However, it could benefit from improved structure and clarity in certain sections, particularly in the skills presentation and project descriptions.',
 'scores': {'structure_and_readability_score': 75,
  'completeness_score': 85,
  'technical_skills_clarity_score': 70,
  'experience_description_score': 80,
  'projects_description_score': 75,
  'measurable_results_score': 60,
  'it_relevance_score': 90},
 'strengths': ['Strong academic performance with a high GPA (9.2/10.0)',
  'Relevant coursework that aligns well with software engineering roles',
  'Diverse project experience showcasing practical application of skills',
  'Clear communication of technical skills and areas of specialization'],
 'weaknesses': ['Skills section lacks organization and clarity, making it hard to quickly assess proficiency',
  'Project descriptions could include measurable outc

## 9. Validate Individual Scores

In [16]:
def clamp_score(value):
    try:
        value = int(value)
    except (ValueError, TypeError):
        value = 0

    return max(0, min(100, value))


for score_name, score_value in cv_quality_dict["scores"].items():
    cv_quality_dict["scores"][score_name] = clamp_score(score_value)

In [17]:
cv_quality_dict["scores"]

{'structure_and_readability_score': 75,
 'completeness_score': 85,
 'technical_skills_clarity_score': 70,
 'experience_description_score': 80,
 'projects_description_score': 75,
 'measurable_results_score': 60,
 'it_relevance_score': 90}

## 10. Calculate Final CV Quality Score

In [18]:
score_weights = {
    "structure_and_readability_score": 0.15,
    "completeness_score": 0.15,
    "technical_skills_clarity_score": 0.20,
    "experience_description_score": 0.20,
    "projects_description_score": 0.15,
    "measurable_results_score": 0.10,
    "it_relevance_score": 0.05
}

final_cv_quality_score = 0


In [19]:
for score_name, weight in score_weights.items():
    final_cv_quality_score += cv_quality_dict["scores"][score_name] * weight

In [20]:
final_cv_quality_score = round(final_cv_quality_score)

In [21]:
final_cv_quality_score

76

## 11. Define CV Quality Category

In [22]:
def get_cv_quality_category(score):
    if score < 50:
        return "Weak CV"
    elif score < 70:
        return "Basic CV"
    elif score < 85:
        return "Good CV"
    else:
        return "Strong CV"


In [23]:
cv_quality_category = get_cv_quality_category(final_cv_quality_score)

In [24]:

cv_quality_category

'Good CV'

## 12. Add Final Score and Category to Result

In [25]:
cv_quality_dict["final_cv_quality_score"] = final_cv_quality_score
cv_quality_dict["cv_quality_category"] = cv_quality_category

In [26]:
cv_quality_dict

{'overall_summary': 'The CV presents a strong candidate profile with a solid educational background and relevant project experience. However, it could benefit from improved structure and clarity in certain sections, particularly in the skills presentation and project descriptions.',
 'scores': {'structure_and_readability_score': 75,
  'completeness_score': 85,
  'technical_skills_clarity_score': 70,
  'experience_description_score': 80,
  'projects_description_score': 75,
  'measurable_results_score': 60,
  'it_relevance_score': 90},
 'strengths': ['Strong academic performance with a high GPA (9.2/10.0)',
  'Relevant coursework that aligns well with software engineering roles',
  'Diverse project experience showcasing practical application of skills',
  'Clear communication of technical skills and areas of specialization'],
 'weaknesses': ['Skills section lacks organization and clarity, making it hard to quickly assess proficiency',
  'Project descriptions could include measurable outc

## 13. Create Score Breakdown Table

In [27]:
score_breakdown_df = pd.DataFrame([
    {
        "criterion": score_name,
        "score": score,
        "weight": score_weights.get(score_name, 0),
        "weighted_score": round(score * score_weights.get(score_name, 0), 2)
    }
    for score_name, score in cv_quality_dict["scores"].items()
])


In [28]:
score_breakdown_df

,criterion,score,weight,weighted_score
0,structure_and_readability_score,75,0.15,11.25
1,completeness_score,85,0.15,12.75
2,technical_skills_clarity_score,70,0.20,14.00
3,experience_description_score,80,0.20,16.00
4,projects_description_score,75,0.15,11.25
5,measurable_results_score,60,0.10,6.00
6,it_relevance_score,90,0.05,4.50


## 14 Generate Markdown Report

In [29]:
def format_bullet_list(items):

    if not items:
        return "- No items identified."

    return "\n".join([f"- {item}" for item in items])


cv_quality_markdown_report = f"""
# CV Quality Analysis Report

## Final CV Quality Score

**Score:** {cv_quality_dict["final_cv_quality_score"]}/100  
**Category:** {cv_quality_dict["cv_quality_category"]}

## Overall Summary

{cv_quality_dict["overall_summary"]}

## Score Breakdown

| Criterion | Score | Weight |
|---|---:|---:|
| Structure and readability | {cv_quality_dict["scores"]["structure_and_readability_score"]} | 15% |
| Completeness | {cv_quality_dict["scores"]["completeness_score"]} | 15% |
| Technical skills clarity | {cv_quality_dict["scores"]["technical_skills_clarity_score"]} | 20% |
| Experience description | {cv_quality_dict["scores"]["experience_description_score"]} | 20% |
| Projects description | {cv_quality_dict["scores"]["projects_description_score"]} | 15% |
| Measurable results | {cv_quality_dict["scores"]["measurable_results_score"]} | 10% |
| IT relevance | {cv_quality_dict["scores"]["it_relevance_score"]} | 5% |

## Strengths

{format_bullet_list(cv_quality_dict["strengths"])}

## Weaknesses

{format_bullet_list(cv_quality_dict["weaknesses"])}

## Missing or Unclear Sections

{format_bullet_list(cv_quality_dict["missing_or_unclear_sections"])}

## CV Improvement Recommendations

{format_bullet_list(cv_quality_dict["cv_improvement_recommendations"])}

## Evidence Notes

{format_bullet_list(cv_quality_dict["evidence_notes"])}
"""


In [30]:
print(cv_quality_markdown_report)


# CV Quality Analysis Report

## Final CV Quality Score

**Score:** 76/100  
**Category:** Good CV

## Overall Summary

The CV presents a strong candidate profile with a solid educational background and relevant project experience. However, it could benefit from improved structure and clarity in certain sections, particularly in the skills presentation and project descriptions.

## Score Breakdown

| Criterion | Score | Weight |
|---|---:|---:|
| Structure and readability | 75 | 15% |
| Completeness | 85 | 15% |
| Technical skills clarity | 70 | 20% |
| Experience description | 80 | 20% |
| Projects description | 75 | 15% |
| Measurable results | 60 | 10% |
| IT relevance | 90 | 5% |

## Strengths

- Strong academic performance with a high GPA (9.2/10.0)
- Relevant coursework that aligns well with software engineering roles
- Diverse project experience showcasing practical application of skills
- Clear communication of technical skills and areas of specialization

## Weaknesses

- Ski

## 15. Save JSON Result & Markdown Report

In [32]:
with open(json_output_path, "w", encoding="utf-8") as file:
    json.dump(cv_quality_dict, file, indent=4, ensure_ascii=False)

with open(markdown_output_path, "w", encoding="utf-8") as file:
    file.write(cv_quality_markdown_report)

In [33]:
print(f"CV quality analysis JSON saved to: {json_output_path}")
print(f"CV quality Markdown report saved to: {markdown_output_path}")

CV quality analysis JSON saved to: ..\outputs\cv_quality\cv_quality_analysis.json
CV quality Markdown report saved to: ..\outputs\cv_quality\cv_quality_report.md
